In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    classification_report,
    precision_recall_curve,
    auc,
)
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch

# --- Ensure data is numeric ---
# If you have categorical variables, encode them first
# For example:
# for col in cat_cols:
#     le = LabelEncoder()
#     X_train[col] = le.fit_transform(X_train[col])
#     X_test[col] = le.transform(X_test[col])

## Paths
INPUT_TRAIN_PATH = Path("../data/train_X.csv")
INPUT_TEST_PATH = Path("../data/test_X.csv")
INPUT_TRAIN_Y_PATH = Path("../data/train_y.csv")
INPUT_TEST_Y_PATH = Path("../data/test_y.csv")
#
# -------------------------------
# Load dataset
# -------------------------------
X_train = pd.read_csv(INPUT_TRAIN_PATH)
print("head of X_train:", X_train.head)
#
y_train = pd.read_csv(INPUT_TRAIN_Y_PATH)
print("head of y_train:", y_train.head)
#
X_test = pd.read_csv(INPUT_TEST_PATH)
print("head of X_test:", X_test.head)
#
y_test = pd.read_csv(INPUT_TEST_Y_PATH)
print("head of y_test:", y_test.head)


# --- Convert to numpy ---
X_train_np = np.array(X_train)
X_test_np = np.array(X_test)
y_train_np = np.ravel(np.array(y_train))
y_test_np = np.ravel(np.array(y_test))

# --- Compute class weights for imbalance ---
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train_np)
weights = compute_class_weight('balanced', classes=classes, y=y_train_np)
class_weights = dict(zip(classes, weights))
print("Class Weights:", class_weights)

# --- Initialize TabNet ---
tabnet_clf = TabNetClassifier(
    n_d=16,                 # width of decision prediction layer
    n_a=16,                 # width of attention embedding
    n_steps=5,              # number of decision steps
    gamma=1.5,              # relaxation parameter
    n_independent=2,
    n_shared=2,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-3),
    mask_type='sparsemax',  # sparsemax or entmax
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    verbose=10,
    seed=42
)
sample_weights = np.array([class_weights[label] for label in y_train_np])

# --- Fit model ---
tabnet_clf.fit(
    X_train=X_train_np, y_train=y_train_np,
    eval_set=[(X_train_np, y_train_np), (X_test_np, y_test_np)],
    eval_name=['train', 'test'],
    eval_metric=['auc', 'accuracy'],
    max_epochs=200,
    patience=20,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    weights=sample_weights
)

# --- Predictions ---
y_pred = tabnet_clf.predict(X_test_np)
y_pred_proba = tabnet_clf.predict_proba(X_test_np)[:, 1]

# --- Metrics ---
roc = roc_auc_score(y_test_np, y_pred_proba)
prec, rec, _ = precision_recall_curve(y_test_np, y_pred_proba)
pr_auc = auc(rec, prec)

results = {
    "Accuracy": accuracy_score(y_test_np, y_pred),
    "ROC-AUC": roc,
    "PR-AUC": pr_auc,
    "F1": f1_score(y_test_np, y_pred),
    "Recall": recall_score(y_test_np, y_pred),
    "Precision": precision_score(y_test_np, y_pred),
}

print("\n=== Metrics ===")
for k, v in results.items():
    print(f"{k:10s}: {v:.4f}")

print("\nClassification Report:")
print(classification_report(y_test_np, y_pred, digits=4))

# --- Feature Importance ---
print("\nFeature Importances:")
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": tabnet_clf.feature_importances_
}).sort_values("importance", ascending=False)
print(feature_importance)

# --- Optional: Save model ---
#tabnet_clf.save_model("tabnet_insurance_renewal.zip")


head of X_train: <bound method NDFrame.head of        perc_premium_paid_by_cash_credit    Income  premium_to_income  \
0                             -0.762252 -1.056313          -0.441069   
1                             -0.851828  0.619890          -0.985752   
2                             -0.481582  0.030873           0.135439   
3                             -0.860785  0.166194           1.061873   
4                              0.040943 -0.099648          -0.110840   
...                                 ...       ...                ...   
63877                          1.333818 -0.459054          -0.645000   
63878                         -0.577129  0.496163           0.912571   
63879                         -0.287501  0.678221          -1.010833   
63880                         -0.938418  0.173893           1.049436   
63881                         -0.935432 -0.613486          -0.368195   

       application_underwriting_score  age_in_days  premiums_paid_ratio  \
0            

c:\Users\ebalkes\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.89284 | train_auc: 0.70843 | train_accuracy: 0.65072 | test_auc: 0.69202 | test_accuracy: 0.64899 |  0:00:06s
epoch 10 | loss: 0.52658 | train_auc: 0.82532 | train_accuracy: 0.77482 | test_auc: 0.81834 | test_accuracy: 0.77397 |  0:01:20s
epoch 20 | loss: 0.51389 | train_auc: 0.83625 | train_accuracy: 0.76723 | test_auc: 0.8306  | test_accuracy: 0.76908 |  0:02:41s

Early stopping occurred at epoch 26 with best_epoch = 6 and best_test_accuracy = 0.79394


c:\Users\ebalkes\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



=== Metrics ===
Accuracy  : 0.7939
ROC-AUC   : 0.8073
PR-AUC    : 0.9806
F1        : 0.8796
Recall    : 0.8030
Precision : 0.9723

Classification Report:
              precision    recall  f1-score   support

           0     0.1824    0.6580    0.2857      1000
           1     0.9723    0.8030    0.8796     14971

    accuracy                         0.7939     15971
   macro avg     0.5774    0.7305    0.5826     15971
weighted avg     0.9229    0.7939    0.8424     15971


Feature Importances:
                             feature  importance
0   perc_premium_paid_by_cash_credit    0.135924
14                sourcing_channel_B    0.120115
6                  total_late_counts    0.114997
7                       age_in_years    0.094568
11             Count_3-6_months_late    0.054604
10                           premium    0.052720
13         residence_area_type_Urban    0.052671
5                premiums_paid_ratio    0.052113
3     application_underwriting_score    0.042960
8     

'tabnet_insurance_renewal.zip.zip'